# Variational Autoencoders (VAE)

_Modern Deep Learning & AI_

**Encode to a distribution, not a point. Sample it, decode it, and you can generate brand-new data.**

A plain autoencoder maps each input to one point in the code space. Gaps between points decode to garbage, so you cannot generate new samples.

     A variational autoencoder fixes this. The encoder outputs a small distribution for each input: a mean $\mu$ and a spread $\sigma$.

---

This notebook builds a VAE from scratch, one step at a time. Run each cell, read the note above it, then explore how real handwritten digits arrange themselves in a learned latent space. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## First, look at the data

Before training on it, see what each example actually contains. These are **images, not table columns** — each sample is an 8x8 grid of pixel intensities (0–16).

In [ ]:
from sklearn.datasets import load_digits

digits = load_digits()
print("image array:", digits.images.shape, " labels:", digits.target.shape)
samples = list(zip(digits.images, digits.target))
fig, axes = plt.subplots(1, 5, figsize=(8, 2))
for ax, (image, label) in zip(axes, samples):
    ax.imshow(image, cmap="gray")
    ax.set_title(str(label))
    ax.axis("off")
plt.show()

## Reference implementation — PyTorch

### Step 1 — Define the VAE network

A VAE's encoder does not map an input to a single point — it maps it to a small Gaussian, described by a mean `mu` and a log-variance `logvar`. We predict the *log* of the variance so that exponentiating it always yields a positive spread. The decoder takes a sampled latent vector and reconstructs the original input.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class VAE(nn.Module):
    def __init__(self, n_in=5, latent=2):
        super().__init__()
        self.enc = nn.Linear(n_in, 8)
        self.mu = nn.Linear(8, latent)
        self.logvar = nn.Linear(8, latent)         # log-variance keeps sigma positive
        self.dec = nn.Sequential(
            nn.Linear(latent, 8),
            nn.ReLU(),
            nn.Linear(8, n_in),
        )

    def reparam(self, mu, logvar):
        std = torch.exp(0.5 * logvar)              # sigma
        eps = torch.randn_like(std)                # noise ~ N(0, I)
        return mu + std * eps                      # z = mu + sigma * eps

    def forward(self, x):
        h = F.relu(self.enc(x))
        mu, logvar = self.mu(h), self.logvar(h)
        z = self.reparam(mu, logvar)
        return self.dec(z), mu, logvar

### Step 2 — Run a synthetic batch forward

Sampling the latent `z` is random, which would normally block gradients. The **reparameterization trick** sidesteps that: instead of sampling `z` directly, we sample fixed noise `eps` and compute `z = mu + sigma·eps`, so gradients still flow through `mu` and `sigma`. We push a small random batch through the model to get a reconstruction plus the distribution parameters.

In [ ]:
torch.manual_seed(0)
model = VAE()

x = torch.rand(8, 5)                                # synthetic batch
x_hat, mu, logvar = model(x)

### Step 3 — Compute the two-part VAE loss

A VAE is trained on two terms. The **reconstruction loss** rewards decoding back to the original input. The **KL divergence** pulls each input's latent distribution toward a standard normal `N(0, I)`, which keeps the latent space smooth and packed together so we can sample from it later. The total loss is simply their sum.

In [ ]:
recon = F.mse_loss(x_hat, x, reduction="sum")
kl = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())   # KL to N(0, I)
loss = recon + kl

print("recon:", round(recon.item(), 2), "kl:", round(kl.item(), 2))

## Visualize the data & results

_In the learned latent of real digits, how do classes cluster, and which digits are hardest to reconstruct?_

### Step 1 — Reduce real digits to a 2-D latent with PCA

To peek at a latent space without training a full VAE, we use PCA as a stand-in encoder/decoder. We load the handwritten digits, scale pixels to `[0, 1]`, project each image down to 2 latent coordinates, and reconstruct it back. The per-image reconstruction error tells us how much detail each digit lost in the squeeze.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.decomposition import PCA

# latent space + per-class reconstruction error on REAL handwritten digits
digits = load_digits()
X = digits.data / 16.0
y = digits.target

pca = PCA(n_components=2)
Z = pca.fit_transform(X)                            # 2-D latent coordinates
recon = pca.inverse_transform(Z)                    # reconstruct from the latent
mse = np.mean((X - recon) ** 2, axis=1)             # per-image reconstruction error

### Step 2 — Scatter the classes in latent space

Plotting a handful of points from digits 0–3 by their two latent coordinates shows whether similar digits land near each other. Well-separated coloured clusters mean the latent space has organised the classes — exactly the smooth, structured geometry a VAE aims for.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for c, col in zip([0, 1, 2, 3], ["#c89bff", "#4ea1ff", "#7ee787", "#ffb454"]):
    idx = np.where(y == c)[0][:15]
    axes[0].scatter(Z[idx, 0], Z[idx, 1], color=col, label="digit %d" % c)
axes[0].set_xlabel("latent z1")
axes[0].set_ylabel("latent z2")
axes[0].set_title("real digits 0-3 in latent space")
axes[0].legend()

### Step 3 — Compare reconstruction error per digit class

Averaging the reconstruction error within each digit class shows which digits are hardest to compress into just two numbers. Taller bars mark classes whose shapes vary the most, so a tiny 2-D latent cannot capture them as faithfully.

In [ ]:
class_mse = [float(mse[y == c].mean()) for c in range(10)]
axes[1].bar([str(c) for c in range(10)], class_mse, color="#c89bff")
axes[1].set_title("mean MSE per digit class")

plt.show()